# LLM-Based Code Smell Detection with Qwen2.5-Coder

This notebook is self-contained. It does not need `code_smell_llm.py`.

Input data expected on Kaggle:

```text
dataset_java/
  ComplexConditional_1
  ComplexConditional_2
  ComplexMethod_1
  ComplexMethod_2
  FeatureEnvy_1
  FeatureEnvy_2
  MultifacetedAbstraction_1
  MultifacetedAbstraction_2
  readme.txt
```

The Java files are split archives. The notebook merges each pair into a `.7z`, samples positive/negative files inside the archive, and extracts only the selected files.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

base_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "py7zr": "py7zr",
}
missing = [pkg for mod, pkg in base_packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("Base dependencies ready")

In [ ]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()
print("Kaggle API OK")

In [ ]:
!find /kaggle/input -maxdepth 6 -type d -name "checkpoint-*" | head -50
!find /kaggle/input -maxdepth 6 -type f -name "adapter_model.safetensors" | head -50

## Imports and Configuration

`FAST_RUN=True` is for smoke testing. It uses a small sample size and should finish much faster.

Run order recommendation:

1. Baseline only.
2. Baseline + Qwen 3B smoke test.
3. Full Qwen 3B.
4. Optional Qwen 7B.

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.svm import LinearSVC

SMELLS = [
    "ComplexMethod",
    "ComplexConditional",
    "FeatureEnvy",
    "MultifacetedAbstraction",
]

SMELL_DEFINITIONS = {
    "ComplexMethod": "A method that is too long, performs too many responsibilities, or is difficult to understand.",
    "ComplexConditional": "A method or code fragment with deeply nested or hard-to-read conditional logic.",
    "FeatureEnvy": "A class or method that uses data or behavior from another class more than from its own class.",
    "MultifacetedAbstraction": "A class abstraction that mixes multiple responsibilities or unrelated concerns.",
}

GITHUB_DATA_URL = "https://raw.githubusercontent.com/tushartushar/DeepLearningSmells/master/data/training_data_java"
JAVA_DIR_NAMES = ("dataset_java", "training_data_java")

@dataclass
class Config:
    work_dir: Path = Path("/kaggle/working/code_smell_llm")
    kaggle_input: Path = Path("/kaggle/input")
    seed: int = 42
    fast_run: bool = True
    max_per_class_per_task: int = 80
    full_max_per_class_per_task: int = 1500
    run_baseline: bool = True
    run_qwen_3b: bool = True
    run_qwen_7b: bool = False
    max_seq_length: int = 2048
    code_char_limit: int = 12000
    num_train_epochs: float = 1.0
    full_num_train_epochs: float = 2.0
    learning_rate: float = 2e-4
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    eval_batch_size: int = 1
    save_steps: int = 200
    save_total_limit: int = 4
    resume_from_checkpoint: bool = True
    checkpoint_selection_metric: str = "mcc"
    checkpoint_selection_tiebreaker: str = "f1"
    checkpoint_selection_max_candidates: int = 3
    qwen_run_mode: str = "train_then_eval"
    checkpoint_backup_enabled: bool = False
    checkpoint_dataset_slug: str = "YOUR_USERNAME/code-smell-qwen-checkpoints"
    checkpoint_backup_dir: Path = Path("/kaggle/working/qwen_checkpoint_backup")
    checkpoint_backup_keep_last: int = 1
    checkpoint_backup_strict: bool = False
    restore_checkpoint_from_dataset: bool = True
    eval_output_upload_enabled: bool = True
    eval_output_upload_strict: bool = False
    qwen_3b: str = "Qwen/Qwen2.5-Coder-3B-Instruct"
    qwen_7b: str = "Qwen/Qwen2.5-Coder-7B-Instruct"

    @property
    def sample_limit(self) -> int:
        return self.max_per_class_per_task if self.fast_run else self.full_max_per_class_per_task

    @property
    def epochs(self) -> float:
        return self.num_train_epochs if self.fast_run else self.full_num_train_epochs

FAST_RUN = True
cfg = Config(
    fast_run=FAST_RUN,
    max_per_class_per_task=80,
    full_max_per_class_per_task=1500,
    run_baseline=True,
    run_qwen_3b=True,
    run_qwen_7b=False,
    max_seq_length=2048,
    code_char_limit=12000,
    num_train_epochs=1.0,
    full_num_train_epochs=2.0,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    eval_batch_size=1,
    save_steps=10 if FAST_RUN else 200,
    save_total_limit=4,
    resume_from_checkpoint=True,
    qwen_run_mode="train_then_eval",
    checkpoint_backup_enabled=False,
    checkpoint_dataset_slug="YOUR_USERNAME/code-smell-qwen-checkpoints",
    checkpoint_backup_keep_last=1,
    checkpoint_selection_max_candidates=3,
    restore_checkpoint_from_dataset=True,
    eval_output_upload_enabled=True,
)
cfg

## Utility Functions

These helpers make the run reproducible and keep all generated files under `/kaggle/working/code_smell_llm`.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def safe_name(text: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in text).strip("_")


set_seed(cfg.seed)
ensure_dir(cfg.work_dir)
print("work_dir:", cfg.work_dir)

## Locate Java Dataset

The notebook searches Kaggle inputs for `dataset_java` or `training_data_java`.
It also supports the case where the eight split files are placed directly inside the Kaggle dataset root.

In [ ]:
def looks_like_java_parts(path: Path) -> bool:
    return path.exists() and all((path / f"{smell}_1").exists() and (path / f"{smell}_2").exists() for smell in SMELLS)


def find_java_parts_dir(cfg: Config) -> Path:
    roots = [Path.cwd(), Path.cwd().parent, cfg.kaggle_input]
    if cfg.kaggle_input.exists():
        roots.extend(sorted(p for p in cfg.kaggle_input.iterdir() if p.is_dir()))

    candidates = []
    for root in roots:
        if not root.exists():
            continue
        candidates.append(root)
        for name in JAVA_DIR_NAMES:
            candidates.append(root / name)
            candidates.append(root / "data" / name)

    for candidate in candidates:
        if looks_like_java_parts(candidate):
            return candidate

    for root in roots:
        if not root.exists():
            continue
        for name in JAVA_DIR_NAMES:
            for candidate in root.rglob(name):
                if looks_like_java_parts(candidate):
                    return candidate

    raise FileNotFoundError(
        "Could not find Java split archives. Add a Kaggle Dataset containing dataset_java/ or training_data_java/."
    )


java_parts_dir = find_java_parts_dir(cfg)
print("Java dataset:", java_parts_dir)
print(sorted(p.name for p in java_parts_dir.iterdir() if p.is_file()))

## Merge Split Archives

The source dataset stores each Java smell archive as two split files. This cell merges each pair into a valid `.7z` archive under the working directory.

In [ ]:
def merge_java_archives(parts_dir: Path, cfg: Config) -> dict[str, Path]:
    out_dir = ensure_dir(cfg.work_dir / "raw_archives" / "merged_java_7z")
    merged = {}
    for smell in SMELLS:
        out_file = out_dir / f"{smell}.7z"
        if not out_file.exists():
            with out_file.open("wb") as dst:
                for part in [parts_dir / f"{smell}_1", parts_dir / f"{smell}_2"]:
                    with part.open("rb") as src:
                        shutil.copyfileobj(src, dst)
        merged[smell] = out_file
        print(f"{smell:28} -> {out_file.name} ({out_file.stat().st_size / 1024 / 1024:.1f} MB)")
    return merged


merged_archives = merge_java_archives(java_parts_dir, cfg)

## Sample Raw Source Files from Archives

Important: the raw archives contain millions of files. Extracting all of them is slow and can overload Kaggle storage.

This notebook only extracts the sampled files required by the experiment.

In [ ]:
def archive_label(name: str) -> int | None:
    normalized = "/" + name.replace("\\", "/")
    if "/Positive/" in normalized:
        return 1
    if "/Negative/" in normalized:
        return 0
    return None


def list_labeled_members(archive_path: Path) -> tuple[list[str], list[str]]:
    import py7zr
    with py7zr.SevenZipFile(archive_path, "r") as zf:
        names = [name for name in zf.getnames() if not name.endswith("/")]
    positive = [name for name in names if archive_label(name) == 1]
    negative = [name for name in names if archive_label(name) == 0]
    return positive, negative


def sample_members(members: list[str], limit: int, seed: int) -> list[str]:
    rng = np.random.default_rng(seed)
    members = np.array(sorted(members), dtype=object)
    if len(members) <= limit:
        return members.tolist()
    idx = rng.choice(len(members), size=limit, replace=False)
    return members[idx].tolist()


def extract_targets(archive_path: Path, targets: list[str], out_dir: Path) -> None:
    import py7zr
    marker = out_dir / ".extract.done"
    if marker.exists():
        return
    ensure_dir(out_dir)
    with py7zr.SevenZipFile(archive_path, "r") as zf:
        zf.extract(path=out_dir, targets=targets)
    marker.write_text("ok\n", encoding="utf-8")


def read_text(path: Path) -> str:
    for encoding in ("utf-8", "latin-1", "cp1252"):
        try:
            return path.read_text(encoding=encoding, errors="ignore")
        except Exception:
            pass
    return path.read_bytes().decode("utf-8", errors="ignore")


def build_dataset_from_archives(archives: dict[str, Path], cfg: Config) -> pd.DataFrame:
    rows = []
    stats = []
    for smell, archive_path in archives.items():
        positive, negative = list_labeled_members(archive_path)
        selected_positive = sample_members(positive, cfg.sample_limit, cfg.seed + len(smell))
        selected_negative = sample_members(negative, cfg.sample_limit, cfg.seed + len(smell) + 1000)
        selected = selected_positive + selected_negative
        extract_dir = cfg.work_dir / "selected_raw_source" / smell
        extract_targets(archive_path, selected, extract_dir)

        stats.append({
            "smell": smell,
            "archive_positive": len(positive),
            "archive_negative": len(negative),
            "selected_positive": len(selected_positive),
            "selected_negative": len(selected_negative),
        })

        for member in selected:
            file_path = extract_dir / member
            code = read_text(file_path).strip()
            if not code:
                continue
            rows.append({
                "language": "Java",
                "smell": smell,
                "smell_definition": SMELL_DEFINITIONS[smell],
                "code": code,
                "label": int(archive_label(member)),
                "source_path": member,
            })

    stats_df = pd.DataFrame(stats)
    display(stats_df)

    df = pd.DataFrame(rows)
    df["code_hash"] = df["code"].map(lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest())
    df = df.drop_duplicates(["smell", "code_hash"]).reset_index(drop=True)
    return df


df = build_dataset_from_archives(merged_archives, cfg)
print("dataset rows after duplicate removal:", len(df))
display(df.groupby(["smell", "label"]).size().unstack(fill_value=0))

## Train / Validation / Test Split

Splitting is stratified within each `smell + label` group. The same split is reused by the baseline and the LLM.

In [ ]:
def add_splits(df: pd.DataFrame, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    out = df.copy()
    out["split"] = "train"
    for _, idx in out.groupby(["smell", "label"]).groups.items():
        idx = np.array(list(idx))
        rng.shuffle(idx)
        n = len(idx)
        if n < 3:
            continue
        n_train = max(1, int(math.floor(n * 0.70)))
        n_val = max(1, int(math.floor(n * 0.15)))
        if n_train + n_val >= n:
            n_val = max(1, n - n_train - 1)
        out.loc[idx[:n_train], "split"] = "train"
        out.loc[idx[n_train:n_train + n_val], "split"] = "validation"
        out.loc[idx[n_train + n_val:], "split"] = "test"
    return out.reset_index(drop=True)


df = add_splits(df, cfg.seed)
display(df.groupby(["smell", "label", "split"]).size().rename("n").reset_index())

## Prompt Format

The LLM receives raw source code plus the target smell definition, then learns to answer only `YES` or `NO`.

In [ ]:
def truncate_code(code: str, limit: int) -> str:
    if len(code) <= limit:
        return code
    half = max(1, (limit - 120) // 2)
    return code[:half] + "\n\n/* ... middle truncated ... */\n\n" + code[-half:]


def make_user_prompt(row, code_char_limit: int = 12000) -> str:
    code = truncate_code(str(row["code"]), code_char_limit)
    return (
        f"Language: Java\n"
        f"Target smell: {row['smell']}\n"
        f"Definition: {row['smell_definition']}\n\n"
        f"Code:\n{code}\n\n"
        "Answer only YES or NO."
    )


def make_baseline_text(row, code_char_limit: int = 20000) -> str:
    code = truncate_code(str(row["code"]), code_char_limit)
    return f"Language: Java\nTarget smell: {row['smell']}\nCode:\n{code}"


print(make_user_prompt(df.iloc[0], 1200)[:1600])

## Metrics

F1 and MCC are important because code smell labels are naturally imbalanced.

In [ ]:
def metric_row(model: str, scope: str, group: str, y_true, y_pred, y_score=None) -> dict:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    row = {
        "model": model,
        "scope": scope,
        "group": group,
        "n": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0,
        "roc_auc": np.nan,
    }
    if y_score is not None and len(np.unique(y_true)) > 1:
        try:
            row["roc_auc"] = roc_auc_score(y_true, y_score)
        except Exception:
            row["roc_auc"] = np.nan
    return row


def evaluate_predictions(preds: pd.DataFrame, model_name: str) -> pd.DataFrame:
    rows = [metric_row(model_name, "overall", "all", preds["label"], preds["prediction"], preds.get("score"))]
    for smell, group in preds.groupby("smell"):
        rows.append(metric_row(model_name, "smell", smell, group["label"], group["prediction"], group.get("score")))
    return pd.DataFrame(rows)

## Baseline: TF-IDF + LinearSVC

This baseline is fast and gives a useful comparison point before running Qwen.

In [ ]:
def run_baseline(df: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame]:
    train = df[df["split"] == "train"]
    test = df[df["split"] == "test"].copy()
    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2, max_features=250_000)
    x_train = vectorizer.fit_transform(train.apply(make_baseline_text, axis=1))
    x_test = vectorizer.transform(test.apply(make_baseline_text, axis=1))
    clf = LinearSVC(class_weight="balanced", random_state=cfg.seed)
    clf.fit(x_train, train["label"])
    scores = clf.decision_function(x_test)
    test["prediction"] = (scores >= 0).astype(int)
    test["score"] = scores
    test["model"] = "TF-IDF + LinearSVC"
    metrics = evaluate_predictions(test, "TF-IDF + LinearSVC")
    return test, metrics


all_predictions = []
all_metrics = []
if cfg.run_baseline:
    baseline_preds, baseline_metrics = run_baseline(df, cfg)
    all_predictions.append(baseline_preds)
    all_metrics.append(baseline_metrics)
    display(baseline_metrics)

## Qwen QLoRA Setup

This section installs and defines the LLM training code. If you only want to test data and baseline, set `run_qwen_3b=False` in the config cell and skip this section.

In [ ]:
def ensure_llm_dependencies() -> None:
    llm_packages = {
        "torch": "torch",
        "transformers": "transformers",
        "accelerate": "accelerate",
        "peft": "peft",
        "bitsandbytes": "bitsandbytes",
    }
    missing = [pkg for mod, pkg in llm_packages.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def chat_prompt(tokenizer, row, cfg: Config) -> str:
    system = "You are a code smell detection model."
    user = make_user_prompt(row, cfg.code_char_limit)
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return system + "\n\n" + user + "\n\nAnswer:"


class PromptAnswerDataset:
    def __init__(self, frame: pd.DataFrame, tokenizer, cfg: Config):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.cfg = cfg
        self.yes_ids = tokenizer(" YES", add_special_tokens=False)["input_ids"]
        self.no_ids = tokenizer(" NO", add_special_tokens=False)["input_ids"]
        self.eos_id = tokenizer.eos_token_id

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        answer_ids = self.yes_ids if int(row["label"]) == 1 else self.no_ids
        answer_ids = answer_ids + [self.eos_id]
        prompt = chat_prompt(self.tokenizer, row, self.cfg)
        prompt_ids = self.tokenizer(
            prompt,
            add_special_tokens=False,
            truncation=True,
            max_length=self.cfg.max_seq_length - len(answer_ids),
        )["input_ids"]
        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids
        return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}


class PromptAnswerCollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = pad_token_id

    def __call__(self, batch):
        import torch
        max_len = max(len(item["input_ids"]) for item in batch)
        input_ids, attention_mask, labels = [], [], []
        for item in batch:
            pad_len = max_len - len(item["input_ids"])
            input_ids.append(item["input_ids"] + [self.pad_token_id] * pad_len)
            attention_mask.append(item["attention_mask"] + [0] * pad_len)
            labels.append(item["labels"] + [-100] * pad_len)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

In [ ]:
def load_qwen_for_qlora(model_id: str, cfg: Config):
    import torch
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    return tokenizer, get_peft_model(model, lora_config)


def make_training_args(output_dir: Path, cfg: Config):
    from transformers import TrainingArguments
    kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=cfg.epochs,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        logging_steps=10,
        save_strategy="steps",
        save_steps=cfg.save_steps,
        save_total_limit=cfg.save_total_limit,
        fp16=True,
        optim="paged_adamw_8bit",
        report_to="none",
        remove_unused_columns=False,
    )
    try:
        return TrainingArguments(evaluation_strategy="no", **kwargs)
    except TypeError:
        return TrainingArguments(eval_strategy="no", **kwargs)


def qwen_model_name(model_id: str) -> str:
    return model_id.split("/")[-1]


def qwen_output_dir(model_id: str, cfg: Config) -> Path:
    return ensure_dir(cfg.work_dir / "checkpoints" / qwen_model_name(model_id))


def checkpoint_dirs(output_dir: Path) -> list[Path]:
    return sorted(
        [path for path in output_dir.glob("checkpoint-*") if path.is_dir()],
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
    )


def valid_checkpoint_dataset_slug(cfg: Config) -> bool:
    slug = str(cfg.checkpoint_dataset_slug).strip()
    return bool(slug and "/" in slug and "YOUR_USERNAME" not in slug and "INSERT_SLUG_HERE" not in slug)


def json_safe_config(cfg: Config) -> dict:
    out = {}
    for key, value in asdict(cfg).items():
        out[key] = str(value) if isinstance(value, Path) else value
    return out


def write_checkpoint_dataset_metadata(cfg: Config) -> None:
    metadata = {
        "id": cfg.checkpoint_dataset_slug,
        "title": cfg.checkpoint_dataset_slug.split("/")[-1].replace("-", " ").title(),
        "subtitle": "QLoRA checkpoints and logs for code smell detection",
        "licenses": [{"name": "CC0-1.0"}],
    }
    cfg.checkpoint_backup_dir.mkdir(parents=True, exist_ok=True)
    (cfg.checkpoint_backup_dir / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")


def copytree_fresh(src: Path, dst: Path) -> None:
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)


def zip_dir(src_dir: Path, zip_path: Path) -> None:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in src_dir.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(src_dir))


def sync_qwen_checkpoint_backup_folder(model_id: str, output_dir: Path, cfg: Config, step: int | None) -> Path | None:
    if not cfg.checkpoint_backup_enabled or not valid_checkpoint_dataset_slug(cfg):
        return None

    model_name = qwen_model_name(model_id)
    safe_model = safe_name(model_name)
    backup_dir = ensure_dir(cfg.checkpoint_backup_dir)
    stage_dir = backup_dir / "_stage" / safe_model
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    ensure_dir(stage_dir)

    keep = max(1, int(cfg.checkpoint_backup_keep_last))
    copied_checkpoints = []
    for checkpoint in checkpoint_dirs(output_dir)[-keep:]:
        dst = stage_dir / "checkpoints" / checkpoint.name
        copytree_fresh(checkpoint, dst)
        copied_checkpoints.append(checkpoint.name)

    final_adapter = output_dir / "adapter"
    has_final_adapter = (final_adapter / "adapter_config.json").exists()
    if has_final_adapter:
        copytree_fresh(final_adapter, stage_dir / "adapter")

    logs_dir = ensure_dir(stage_dir / "logs")
    for pattern in ["training_log*.csv", "training_log*.json"]:
        for log_path in output_dir.glob(pattern):
            shutil.copy2(log_path, logs_dir / log_path.name)
    outputs_dir = cfg.work_dir / "outputs"
    if outputs_dir.exists():
        for log_path in outputs_dir.glob(f"training_log*{safe_model}*"):
            shutil.copy2(log_path, logs_dir / log_path.name)
            shutil.copy2(log_path, backup_dir / log_path.name)

    manifest = {
        "model_id": model_id,
        "model_name": model_name,
        "step": step,
        "created_at_unix": time.time(),
        "output_dir": str(output_dir),
        "copied_checkpoints": copied_checkpoints,
        "has_final_adapter": has_final_adapter,
        "config": json_safe_config(cfg),
    }
    (stage_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    (backup_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    write_checkpoint_dataset_metadata(cfg)

    bundle_path = backup_dir / f"checkpoint_bundle_{safe_model}.zip"
    zip_dir(stage_dir, bundle_path)
    return bundle_path


def upload_qwen_checkpoint_backup(model_id: str, output_dir: Path, cfg: Config, step: int | None, note: str) -> None:
    if not cfg.checkpoint_backup_enabled:
        return
    if not valid_checkpoint_dataset_slug(cfg):
        print("checkpoint backup skipped: set cfg.checkpoint_dataset_slug to '<username>/<dataset-slug>'")
        return
    bundle_path = sync_qwen_checkpoint_backup_folder(model_id, output_dir, cfg, step)
    if bundle_path is None:
        return
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        api.dataset_create_version(
            folder=str(cfg.checkpoint_backup_dir),
            version_notes=note,
            quiet=True,
            convert_to_csv=False,
            dir_mode="skip",
        )
        print("checkpoint backup uploaded:", bundle_path.name)
    except Exception as exc:
        print("checkpoint backup failed:", type(exc).__name__, exc)
        if cfg.checkpoint_backup_strict:
            raise


def sync_eval_outputs_backup_folder(cfg: Config, outputs_dir: Path) -> Path | None:
    if not getattr(cfg, "eval_output_upload_enabled", False) or not valid_checkpoint_dataset_slug(cfg):
        return None
    outputs_dir = Path(outputs_dir)
    if not outputs_dir.exists():
        return None

    backup_dir = ensure_dir(cfg.checkpoint_backup_dir)
    stage_dir = backup_dir / "_eval_outputs"
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    shutil.copytree(outputs_dir, stage_dir)

    eval_manifest = {
        "kind": "evaluation_outputs",
        "created_at_unix": time.time(),
        "outputs_dir": str(outputs_dir),
        "config": json_safe_config(cfg),
    }
    (stage_dir / "eval_manifest.json").write_text(json.dumps(eval_manifest, indent=2), encoding="utf-8")
    (backup_dir / "eval_manifest.json").write_text(json.dumps(eval_manifest, indent=2), encoding="utf-8")

    for filename in [
        "metrics.csv",
        "predictions.csv",
        "checkpoint_validation_metrics.csv",
        "best_checkpoints.csv",
        "dataset_index.csv",
    ]:
        src_file = outputs_dir / filename
        if src_file.exists():
            shutil.copy2(src_file, backup_dir / filename)

    write_checkpoint_dataset_metadata(cfg)
    bundle_path = backup_dir / "eval_outputs_bundle.zip"
    zip_dir(stage_dir, bundle_path)
    return bundle_path


def upload_eval_outputs_backup(cfg: Config, outputs_dir: Path, note: str = "evaluation outputs") -> None:
    if not getattr(cfg, "eval_output_upload_enabled", False):
        return
    if not valid_checkpoint_dataset_slug(cfg):
        print("eval output upload skipped: set cfg.checkpoint_dataset_slug to '<username>/<dataset-slug>'")
        return
    bundle_path = sync_eval_outputs_backup_folder(cfg, outputs_dir)
    if bundle_path is None:
        print("eval output upload skipped: no outputs found")
        return
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        api.dataset_create_version(
            folder=str(cfg.checkpoint_backup_dir),
            version_notes=note,
            quiet=True,
            convert_to_csv=False,
            dir_mode="skip",
        )
        print("eval outputs uploaded:", bundle_path.name)
    except Exception as exc:
        print("eval output upload failed:", type(exc).__name__, exc)
        if getattr(cfg, "eval_output_upload_strict", False):
            raise


def extract_nested_zips(root: Path) -> None:
    for zip_path in list(root.rglob("*.zip")):
        target = zip_path.with_suffix("")
        ensure_dir(target)
        try:
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(target)
        except Exception as exc:
            print("could not extract", zip_path, type(exc).__name__, exc)


def restore_qwen_checkpoints_from_backup(model_id: str, output_dir: Path, cfg: Config) -> bool:
    if not cfg.restore_checkpoint_from_dataset:
        return False

    safe_model = safe_name(qwen_model_name(model_id))
    restore_root = ensure_dir(cfg.work_dir / "restored_checkpoint_backup" / safe_model)

    search_roots = []
    if cfg.kaggle_input.exists():
        search_roots.append(cfg.kaggle_input)

    if valid_checkpoint_dataset_slug(cfg):
        try:
            from kaggle.api.kaggle_api_extended import KaggleApi
            api = KaggleApi()
            api.authenticate()
            api.dataset_download_files(cfg.checkpoint_dataset_slug, path=str(restore_root), quiet=False, unzip=True)
            search_roots.append(restore_root)
        except Exception as exc:
            print("checkpoint dataset download skipped/failed:", type(exc).__name__, exc)

    bundle_stem = f"checkpoint_bundle_{safe_model}"
    bundle_zip = f"{bundle_stem}.zip"
    for root in search_roots:
        for bundle_path in root.rglob(bundle_zip):
            with zipfile.ZipFile(bundle_path) as zf:
                zf.extractall(restore_root)
            search_roots.append(restore_root)
            break

    candidate_roots = [restore_root]
    for root in search_roots:
        candidate_roots.extend(path for path in root.rglob(bundle_stem) if path.is_dir())
        candidate_roots.extend(path.parent for path in root.rglob("checkpoints") if path.is_dir())

    restored = False
    best_seen_step = -1
    for candidate_root in candidate_roots:
        checkpoint_parent = candidate_root / "checkpoints"
        if checkpoint_parent.exists():
            for checkpoint in checkpoint_parent.glob("checkpoint-*"):
                if checkpoint.is_dir():
                    step = int(checkpoint.name.split("-")[-1]) if checkpoint.name.split("-")[-1].isdigit() else -1
                    copytree_fresh(checkpoint, output_dir / checkpoint.name)
                    best_seen_step = max(best_seen_step, step)
                    restored = True

        adapter_dir = candidate_root / "adapter"
        if (adapter_dir / "adapter_config.json").exists():
            copytree_fresh(adapter_dir, output_dir / "adapter")
            restored = True

        logs_dir = candidate_root / "logs"
        if logs_dir.exists():
            outputs_dir = ensure_dir(cfg.work_dir / "outputs")
            for log_path in logs_dir.glob("training_log*"):
                shutil.copy2(log_path, output_dir / log_path.name)
                shutil.copy2(log_path, outputs_dir / log_path.name)

    if restored:
        print("restored Qwen checkpoint backup to", output_dir, "latest step", best_seen_step)
    else:
        print("no restorable Qwen checkpoints found in attached/input backup datasets")
    return restored


def latest_resume_checkpoint(model_id: str, output_dir: Path, cfg: Config) -> str | None:
    if cfg.resume_from_checkpoint and not checkpoint_dirs(output_dir):
        restore_qwen_checkpoints_from_backup(model_id, output_dir, cfg)
    checkpoints = checkpoint_dirs(output_dir)
    if checkpoints:
        print("Resuming from", checkpoints[-1])
        return str(checkpoints[-1])
    return None


def make_live_backup_callback(model_id: str, output_dir: Path, cfg: Config):
    from transformers import TrainerCallback

    class LiveTrainingBackupCallback(TrainerCallback):
        def __init__(self):
            self.rows = []
            self.model_name = qwen_model_name(model_id)
            self.safe_model = safe_name(self.model_name)
            self.live_paths = [
                output_dir / "training_log_live.csv",
                ensure_dir(cfg.work_dir / "outputs") / f"training_log_live_{self.safe_model}.csv",
            ]

        def write_live_log(self):
            if not self.rows:
                return
            frame = pd.DataFrame(self.rows)
            for path in self.live_paths:
                frame.to_csv(path, index=False)

        def on_log(self, args, state, control, logs=None, **kwargs):
            if not logs:
                return
            row = {"step": int(state.global_step), "epoch": state.epoch}
            row.update({key: value for key, value in logs.items() if isinstance(value, (int, float))})
            self.rows.append(row)
            self.write_live_log()

        def on_save(self, args, state, control, **kwargs):
            step = int(state.global_step)
            self.write_live_log()
            upload_qwen_checkpoint_backup(
                model_id,
                output_dir,
                cfg,
                step,
                f"{qwen_model_name(model_id)} checkpoint step {step}",
            )

    return LiveTrainingBackupCallback()


def train_qwen(model_id: str, df: pd.DataFrame, cfg: Config):
    ensure_llm_dependencies()
    from transformers import Trainer

    tokenizer, model = load_qwen_for_qlora(model_id, cfg)
    train_df = df[df["split"] == "train"].reset_index(drop=True)
    train_dataset = PromptAnswerDataset(train_df, tokenizer, cfg)
    collator = PromptAnswerCollator(tokenizer.pad_token_id)

    model_name = qwen_model_name(model_id)
    output_dir = qwen_output_dir(model_id, cfg)
    resume_checkpoint = latest_resume_checkpoint(model_id, output_dir, cfg)

    callback = make_live_backup_callback(model_id, output_dir, cfg)
    trainer = Trainer(
        model=model,
        args=make_training_args(output_dir, cfg),
        train_dataset=train_dataset,
        data_collator=collator,
        callbacks=[callback],
    )
    try:
        trainer.train(resume_from_checkpoint=resume_checkpoint)
    except RuntimeError as exc:
        message = str(exc).lower()
        if "out of memory" in message or ("cuda" in message and "memory" in message):
            raise RuntimeError(
                "CUDA OOM. Retry with max_seq_length=1024, code_char_limit=6000, "
                "per_device_train_batch_size=1, gradient_accumulation_steps=16, run_qwen_7b=False."
            ) from exc
        raise

    log_history = pd.DataFrame(trainer.state.log_history)
    if not log_history.empty:
        outputs_dir = ensure_dir(cfg.work_dir / "outputs")
        log_csv = outputs_dir / f"training_log_{safe_name(model_name)}.csv"
        log_json = outputs_dir / f"training_log_{safe_name(model_name)}.json"
        log_history.to_csv(log_csv, index=False)
        log_history.to_json(log_json, orient="records", indent=2)
        log_history.to_csv(output_dir / "training_log.csv", index=False)
        log_history.to_json(output_dir / "training_log.json", orient="records", indent=2)
        print("training log:", log_csv)

    adapter_dir = output_dir / "adapter"
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    upload_qwen_checkpoint_backup(
        model_id,
        output_dir,
        cfg,
        adapter_step(adapter_dir, output_dir) if "adapter_step" in globals() else None,
        f"{model_name} final adapter",
    )

    del trainer, model, tokenizer
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass
    return output_dir, adapter_dir


In [ ]:
def load_qwen_adapter_for_inference(model_id: str, adapter_path: Path, cfg: Config):
    ensure_llm_dependencies()
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    base_model.config.use_cache = False
    model = PeftModel.from_pretrained(base_model, str(adapter_path))
    model.eval()
    return tokenizer, model


def cleanup_qwen_model() -> None:
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass


def adapter_step(adapter_path: Path, output_dir: Path) -> int:
    if adapter_path.name.startswith("checkpoint-"):
        suffix = adapter_path.name.split("-")[-1]
        return int(suffix) if suffix.isdigit() else -1
    log_path = output_dir / "training_log.csv"
    if log_path.exists():
        log_df = pd.read_csv(log_path)
        if "step" in log_df and log_df["step"].notna().any():
            return int(log_df["step"].dropna().max())
    checkpoint_steps = [adapter_step(path, output_dir) for path in output_dir.glob("checkpoint-*")]
    return max(checkpoint_steps) if checkpoint_steps else -1


def list_qwen_adapter_candidates(output_dir: Path) -> list[Path]:
    candidates = [
        path for path in output_dir.glob("checkpoint-*")
        if path.is_dir() and (path / "adapter_config.json").exists()
    ]
    final_adapter = output_dir / "adapter"
    if (final_adapter / "adapter_config.json").exists():
        candidates.append(final_adapter)
    return sorted(candidates, key=lambda path: (adapter_step(path, output_dir), path.name))


def upsert_csv_by_value(path: Path, frame: pd.DataFrame, column: str, value: str) -> None:
    ensure_dir(path.parent)
    if path.exists():
        existing = pd.read_csv(path)
        if column in existing.columns:
            existing = existing[existing[column] != value]
            frame = pd.concat([existing, frame], ignore_index=True)
    frame.to_csv(path, index=False)


def predict_qwen(model, tokenizer, test_df: pd.DataFrame, cfg: Config, model_label: str) -> pd.DataFrame:
    import torch
    from torch.nn import functional as F
    from tqdm.auto import tqdm

    yes_id = tokenizer(" YES", add_special_tokens=False)["input_ids"][0]
    no_id = tokenizer(" NO", add_special_tokens=False)["input_ids"][0]
    rows = test_df.reset_index(drop=True)
    preds, scores = [], []

    model.eval()
    for start in tqdm(range(0, len(rows), cfg.eval_batch_size), desc=f"predict {model_label}"):
        batch = rows.iloc[start:start + cfg.eval_batch_size]
        prompts = [chat_prompt(tokenizer, row, cfg) for _, row in batch.iterrows()]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=cfg.max_seq_length,
            add_special_tokens=False,
        ).to(model.device)
        with torch.no_grad():
            logits = model(**inputs).logits
        last_positions = inputs["attention_mask"].sum(dim=1) - 1
        next_logits = logits[torch.arange(logits.shape[0], device=logits.device), last_positions]
        binary_logits = next_logits[:, [no_id, yes_id]].float()
        probs = F.softmax(binary_logits, dim=-1)[:, 1].detach().cpu().numpy()
        scores.extend(probs.tolist())
        preds.extend((probs >= 0.5).astype(int).tolist())

    out = rows.copy()
    out["prediction"] = preds
    out["score"] = scores
    out["model"] = model_label
    return out


def select_best_qwen_checkpoint(model_id: str, output_dir: Path, df: pd.DataFrame, cfg: Config) -> Path:
    valid_df = df[df["split"] == "validation"].copy()
    if valid_df.empty:
        raise ValueError("Validation split is empty; cannot select the best Qwen checkpoint.")

    base_label = qwen_model_name(model_id) + " QLoRA"
    if not list_qwen_adapter_candidates(output_dir):
        restore_qwen_checkpoints_from_backup(model_id, output_dir, cfg)
    candidates = list_qwen_adapter_candidates(output_dir)
    if cfg.checkpoint_selection_max_candidates and cfg.checkpoint_selection_max_candidates > 0:
        candidates = candidates[-int(cfg.checkpoint_selection_max_candidates):]
    if not candidates:
        raise FileNotFoundError(f"No LoRA adapter candidates found in {output_dir}")

    all_candidate_metrics = []
    for adapter_path in candidates:
        step = adapter_step(adapter_path, output_dir)
        candidate_label = f"{base_label} {adapter_path.name}"
        print("validating", candidate_label, "step", step)
        tokenizer, model = load_qwen_adapter_for_inference(model_id, adapter_path, cfg)
        valid_preds = predict_qwen(model, tokenizer, valid_df, cfg, candidate_label)
        candidate_metrics = evaluate_predictions(valid_preds, candidate_label)
        candidate_metrics.insert(0, "base_model", base_label)
        candidate_metrics.insert(1, "candidate", adapter_path.name)
        candidate_metrics.insert(2, "candidate_step", step)
        candidate_metrics.insert(3, "candidate_adapter_path", str(adapter_path))
        all_candidate_metrics.append(candidate_metrics)
        del model, tokenizer
        cleanup_qwen_model()

    validation_metrics = pd.concat(all_candidate_metrics, ignore_index=True)
    outputs_dir = ensure_dir(cfg.work_dir / "outputs")
    upsert_csv_by_value(outputs_dir / "checkpoint_validation_metrics.csv", validation_metrics, "base_model", base_label)

    metric = cfg.checkpoint_selection_metric
    tiebreaker = cfg.checkpoint_selection_tiebreaker
    overall = validation_metrics[validation_metrics["scope"] == "overall"].copy()
    if metric not in overall.columns or tiebreaker not in overall.columns:
        raise ValueError(f"Unknown checkpoint selection metric: {metric}/{tiebreaker}")
    overall[metric] = overall[metric].fillna(-np.inf)
    overall[tiebreaker] = overall[tiebreaker].fillna(-np.inf)
    overall = overall.sort_values([metric, tiebreaker, "candidate_step"], ascending=[False, False, False])
    best = overall.iloc[0]

    best_row = pd.DataFrame([{
        "model": base_label,
        "best_adapter_path": best["candidate_adapter_path"],
        "best_step": int(best["candidate_step"]),
        "validation_mcc": float(best["mcc"]),
        "validation_f1": float(best["f1"]),
        "selection_metric": metric,
        "selection_tiebreaker": tiebreaker,
    }])
    upsert_csv_by_value(outputs_dir / "best_checkpoints.csv", best_row, "model", base_label)
    print("best checkpoint:", best["candidate_adapter_path"])
    display(best_row)
    return Path(best["candidate_adapter_path"])


def evaluate_qwen_model(model_id: str, df: pd.DataFrame, cfg: Config):
    output_dir = qwen_output_dir(model_id, cfg)
    if not list_qwen_adapter_candidates(output_dir):
        restore_qwen_checkpoints_from_backup(model_id, output_dir, cfg)
    best_adapter_path = select_best_qwen_checkpoint(model_id, output_dir, df, cfg)

    test_df = df[df["split"] == "test"].copy()
    model_label = qwen_model_name(model_id) + " QLoRA best-val-MCC"
    tokenizer, model = load_qwen_adapter_for_inference(model_id, best_adapter_path, cfg)
    preds = predict_qwen(model, tokenizer, test_df, cfg, model_label)
    metrics = evaluate_predictions(preds, model_label)
    del model, tokenizer
    cleanup_qwen_model()
    return preds, metrics


def run_qwen_model(model_id: str, df: pd.DataFrame, cfg: Config):
    mode = cfg.qwen_run_mode
    if mode not in {"train_only", "eval_only", "train_then_eval"}:
        raise ValueError("cfg.qwen_run_mode must be 'train_only', 'eval_only', or 'train_then_eval'")

    if mode in {"train_only", "train_then_eval"}:
        train_qwen(model_id, df, cfg)

    if mode == "train_only":
        print("train_only complete. Run again with cfg.qwen_run_mode='eval_only' to select checkpoint and test.")
        return pd.DataFrame(), pd.DataFrame()

    return evaluate_qwen_model(model_id, df, cfg)


## Run Qwen Experiments

Start with 3B. Enable 7B only after 3B works.

In [ ]:
selected_models = []
if cfg.run_qwen_3b:
    selected_models.append(cfg.qwen_3b)
if cfg.run_qwen_7b:
    selected_models.append(cfg.qwen_7b)

for model_id in selected_models:
    qwen_preds, qwen_metrics = run_qwen_model(model_id, df, cfg)
    if not qwen_preds.empty:
        all_predictions.append(qwen_preds)
    if not qwen_metrics.empty:
        all_metrics.append(qwen_metrics)
        display(qwen_metrics)

## Save Outputs

These files are the ones to download from Kaggle after the run.

In [ ]:
def write_report_tables(metrics_df: pd.DataFrame, out_dir: Path) -> None:
    table_dir = ensure_dir(out_dir / "report_tables")
    for scope in ["overall", "smell"]:
        table = metrics_df[metrics_df["scope"] == scope].copy()
        if table.empty:
            continue
        for col in ["accuracy", "precision", "recall", "f1", "mcc", "roc_auc"]:
            table[col] = table[col].map(lambda value: "" if pd.isna(value) else f"{value:.3f}")
        table.to_latex(table_dir / f"{scope}_metrics.tex", index=False, escape=True)


def write_confusion_matrices(predictions_df: pd.DataFrame, out_dir: Path) -> None:
    import matplotlib.pyplot as plt
    cm_dir = ensure_dir(out_dir / "confusion_matrices")
    for model_name, group in predictions_df.groupby("model"):
        display_obj = ConfusionMatrixDisplay.from_predictions(
            group["label"],
            group["prediction"],
            display_labels=["NO", "YES"],
            cmap="Blues",
            values_format="d",
        )
        display_obj.ax_.set_title(model_name)
        plt.tight_layout()
        plt.savefig(cm_dir / f"{safe_name(model_name)}.png", dpi=160)
        plt.close()


outputs_dir = ensure_dir(cfg.work_dir / "outputs")
predictions_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
metrics_df = pd.concat(all_metrics, ignore_index=True) if all_metrics else pd.DataFrame()

df.to_csv(outputs_dir / "dataset_index.csv", index=False)
predictions_df.to_csv(outputs_dir / "predictions.csv", index=False)
metrics_df.to_csv(outputs_dir / "metrics.csv", index=False)
write_report_tables(metrics_df, outputs_dir)
if not predictions_df.empty:
    write_confusion_matrices(predictions_df, outputs_dir)

if not metrics_df.empty or not predictions_df.empty:
    upload_eval_outputs_backup(cfg, outputs_dir, "Qwen evaluation outputs")

print("outputs_dir:", outputs_dir)
display(metrics_df)

## Optional: Recover Training Loss from a Failed Notebook

Use this only outside Kaggle if a failed `.ipynb` still contains the Trainer loss table in its output. It cannot recover model weights.


In [ ]:
def recover_training_log_from_failed_notebook(notebook_path: str | Path, output_csv: str | Path) -> pd.DataFrame:
    import re
    source_path = Path(notebook_path)
    nb_failed = json.loads(source_path.read_text(encoding="utf-8"))
    pairs = []
    for cell in nb_failed.get("cells", []):
        for output in cell.get("outputs", []):
            data = output.get("data", {})
            html = data.get("text/html")
            if isinstance(html, list):
                html = "".join(html)
            if not html or "Training Loss" not in html:
                continue
            text = re.sub(r"<[^>]+>", " ", html)
            numbers = re.findall(r"(?<![\d.])(\d{1,6})\s+([0-9]+\.[0-9]+)(?![\d.])", text)
            pairs.extend((int(step), float(loss)) for step, loss in numbers)
    frame = pd.DataFrame(pairs, columns=["step", "loss"]).drop_duplicates("step").sort_values("step")
    output_path = Path(output_csv)
    ensure_dir(output_path.parent)
    frame.to_csv(output_path, index=False)
    return frame

# Example:
# recovered = recover_training_log_from_failed_notebook(
#     r"C:/Users/admin/Downloads/llm-based-code-smell-detection-with-qwen2-5-coder.ipynb",
#     cfg.work_dir / "outputs" / "recovered_training_log_failed_run.csv",
# )
# display(recovered.tail())


## Visualize Training Curves

This cell reads exported `training_log_*.csv` files and saves loss curves for the report.

In [ ]:
import matplotlib.pyplot as plt

curve_dir = ensure_dir(cfg.work_dir / "outputs" / "training_curves")
for log_file in sorted((cfg.work_dir / "outputs").glob("training_log_*.csv")):
    log_df = pd.read_csv(log_file)
    if "step" not in log_df.columns:
        continue
    plt.figure(figsize=(8, 4))
    plotted = False
    if "loss" in log_df.columns:
        train_log = log_df.dropna(subset=["loss"])
        if not train_log.empty:
            plt.plot(train_log["step"], train_log["loss"], label="train loss")
            plotted = True
    if "eval_loss" in log_df.columns:
        eval_log = log_df.dropna(subset=["eval_loss"])
        if not eval_log.empty:
            plt.plot(eval_log["step"], eval_log["eval_loss"], label="validation loss")
            plotted = True
    if plotted:
        plt.xlabel("step")
        plt.ylabel("loss")
        plt.title(log_file.stem.replace("training_log_", ""))
        plt.legend()
        plt.tight_layout()
        out_png = curve_dir / f"{log_file.stem}.png"
        plt.savefig(out_png, dpi=160)
        plt.show()
        print("saved", out_png)
    plt.close()

## Checkpoint Notes

Checkpoints are saved under:

```text
/kaggle/working/code_smell_llm/checkpoints/
```

If Kaggle restarts in the same session, training resumes from the latest `checkpoint-*` folder.
If the session is gone, save the checkpoint folder as Kaggle output/dataset and attach it to the next run.